<h3>Test drought events count, intensity and duration</h3><br><p>Tests the function used to compute drought events length and duration using a odf sheet with a simplified dataset that can be checked by hand</p>

In [12]:
import pandas as pd
import numpy as np
from loguru import logger

In [21]:
spei_test = '/home/politti/data/Danube_5min/spei_nuts_2/test_data/spei_3_nuts_2.ods'
#df = pd.read_csv(spei_csv)
df = pd.read_excel(spei_test, engine="odf", sheet_name="spei3")
gcms = ['GFDL-ESM4', 'MPI-ESM1-2-HR']
scenario = 'historical'
years = list(range(2000, 2011))
dry_col = 'spei3_dry'
df = df[(df['scenario'] == scenario) & (df['year'].isin(years)) & (df['gcm'].isin(gcms))]
df.tail()

,date,year,month,scenario,gcm,spei3,spei3_dry,event_id
259,2010-08-31 00:00:00,2010,8,historical,MPI-ESM1-2-HR,0.68,0,NaN
260,2010-09-30 00:00:00,2010,9,historical,MPI-ESM1-2-HR,0.80,0,NaN
261,2010-10-31 00:00:00,2010,10,historical,MPI-ESM1-2-HR,1.24,0,NaN
262,2010-11-30 00:00:00,2010,11,historical,MPI-ESM1-2-HR,1.05,0,NaN
263,2010-12-31 00:00:00,2010,12,historical,MPI-ESM1-2-HR,0.91,0,NaN


In [22]:
df['spei3_dry'].max()

1

In [19]:
def compute_drought_events(df, dry_col = 'spei3_dry', spei_col = 'spei3'):
    """
    Computes the length and intensity of drought events for each GCM
    using a single-month pooling strategy.
    """
    # Ensure the dataframe is chronologically sorted per GCM
    df = df.sort_values(by=['gcm', 'year', 'month']).reset_index(drop=True)

    all_gcm_events = []

    # Process each GCM group independently
    for gcm_name, group in df.groupby('gcm', sort=False):
        group = group.copy()

        # 1. Identify drought months (handles both boolean and 1/0 representation)
        is_dry = (group[dry_col] == 1) | (group[dry_col] == True)

        # 2. Apply pooling strategy: Include a non-drought month if it is sandwiched between two drought months
        pooled_dry = is_dry | (is_dry.shift(1).fillna(False) & is_dry.shift(-1).fillna(False))

        if not pooled_dry.any():
            logger.warning(f'No drought events found for GCM: {gcm_name}')
            continue

        # 3. Define unique consecutive event IDs within this GCM
        # An event starts when a month is dry/pooled-dry, but the previous month was not
        event_start = pooled_dry & (~pooled_dry.shift(1).fillna(False))
        group['event_id'] = event_start.cumsum()

        # 4. Filter out non-drought records
        drought_records = group[pooled_dry]

        # 5. Aggregate metrics per event
        event_summary = drought_records.groupby('event_id').agg(
            year=('year', 'first'),
            start_month=('month', 'first'),
            end_month=('month', 'last'),
            event_len=(spei_col, 'count'),  # Total consecutive months inside the pooled event
            event_intensity=(spei_col, 'sum')  # Cumulative sum of SPEI3 inside the pooled event
        ).reset_index()

        # Add the GCM context column
        event_summary['gcm'] = gcm_name

        # Reorder columns to match specifications exactly
        event_summary = event_summary[[
            'year', 'start_month', 'end_month', 'event_id', 'gcm', 'event_len', 'event_intensity'
        ]]

        all_gcm_events.append(event_summary)

    # Combine results from all GCMs into a single DataFrame
    if all_gcm_events:
        return pd.concat(all_gcm_events, ignore_index=True)
    else:
        return pd.DataFrame(
            columns=['year', 'start_month', 'end_month', 'event_id', 'gcm', 'event_len', 'event_intensity'])


In [23]:
df_events = compute_drought_events(df)
df_events.head(11)

/tmp/ipykernel_11166/609001043.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pooled_dry = is_dry | (is_dry.shift(1).fillna(False) & is_dry.shift(-1).fillna(False))
/tmp/ipykernel_11166/609001043.py:27: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  event_start = pooled_dry & (~pooled_dry.shift(1).fillna(False))
/tmp/ipykernel_11166/609001043.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavio

,year,start_month,end_month,event_id,gcm,event_len,event_intensity
0,2001,1,4,1,GFDL-ESM4,4,-3.18
1,2002,1,1,2,GFDL-ESM4,1,-1.21
2,2003,1,3,3,GFDL-ESM4,3,-1.14
3,2002,3,3,1,MPI-ESM1-2-HR,1,-2.17
4,2005,4,5,2,MPI-ESM1-2-HR,2,-4.00
